# T03 — Animations

## What this notebook produces

Three animations, all driven by **pyGMT** for the per-frame rendering and `ffmpeg` (via `subprocess`) for the cross-platform MP4 encoding.

1. **`plate_motion.gif`** — quick-look animated GIF of the global Zahirovic 2022 reconstruction from 250 Ma to 0 Ma. Assembled with `imageio` for trivial drop-in to a README / Slack / Twitter.
2. **`plate_motion.mp4`** — same content, encoded as a cross-platform H.264 MP4 (libx264 + yuv420p + faststart). Plays on QuickTime, Windows Media, every modern browser.
3. **`litho_thickness.mp4`** — a rotating-globe view of global lithospheric thickness from Afonso et al. (2019, *GJI* [doi:10.1093/gji/ggz094](https://doi.org/10.1093/gji/ggz094)), draped on an orthographic projection. **East–West rotation** around the polar axis: viewing longitude sweeps 0° → 360°. Cubehelix colour palette, white coastlines, horizontal colourbar.
4. **`litho_thickness_reconstructed.mp4`** — the same Afonso 2019 lithospheric-thickness grid, but this time **reconstructed through deep time** using GPlately. We build a `gplately.Raster`, partition it with Zahirovic 2022 static polygons, and apply each plate's finite rotation to roll the grid back to its paleo-position at every 2-Myr step from 0 to 60 Ma. Robinson global view, dem4 CPT, reconstructed coastlines overlaid in white at each frame's age.

## Preview frames

Each video section starts with a *preview* — one representative frame rendered inline via `fig.show()`. That lets you sanity-check the recipe (palette, projection, shading, coastline weight) before committing to the full save-loop + ffmpeg encode. Each render loop simply re-calls the same helper function used by the preview, so whatever you see in the preview is what every frame in the MP4 looks like.

## A note on `gmt movie`

GMT itself ships a `movie` module with built-in time labels, progress bars and fade in/out. pyGMT does **not** yet wrap it as a `Figure` method ([upstream issue #364](https://github.com/GenericMappingTools/pygmt/issues/364)) — the canonical Python pattern is to drive it via `subprocess.run(['gmt', 'movie', main_script, …])` exactly the same way we drive ffmpeg below. For the four output files in this notebook we stay in pure pyGMT for the frame rendering (which keeps you inside the Python ecosystem) and use ffmpeg for the MP4 encoding.

**Audience**: both.  
**Difficulty**: ★★☆.  
**Runtime**: ≈5 minutes for all four output files at 720p.

In [ ]:
# Cell 1 — imports
import subprocess
from pathlib import Path
import numpy as np
import xarray as xr
import gplately
import pygmt
from plate_model_manager import PlateModelManager
from IPython.display import display, HTML

# Suite-wide tutorial style: bigger panel titles.
pygmt.config(FONT_TITLE="18p", FONT_LABEL="18p",
             FONT_ANNOT_PRIMARY="14p")

FRAMES_PMM       = Path("./pmm_frames");          FRAMES_PMM.mkdir(exist_ok=True)
FRAMES_LITHO     = Path("./litho_frames");        FRAMES_LITHO.mkdir(exist_ok=True)
FRAMES_LITHO_RC  = Path("./litho_recon_frames");  FRAMES_LITHO_RC.mkdir(exist_ok=True)
VIDEOS           = Path("./videos");              VIDEOS.mkdir(exist_ok=True)

DATA_CANDIDATES = [Path("./data"), Path("../data")]
DATA_ROOT = next((d for d in DATA_CANDIDATES if d.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError("Cannot find ./data or ../data — run Jupyter from the project root or Notebooks/.")
print(f"Using DATA_ROOT = {DATA_ROOT.resolve()}")

In [ ]:
# Cell 2 — plate-motion setup + preview frame
#
# Factor the per-frame rendering into a helper so we can use it for
# (1) one preview frame shown inline below and (2) the save-loop in
# the next cell. Naming convention: frame_0000.png is the FIRST
# (oldest) frame — alphabetical order then equals chronological order
# from past → present, which is what ffmpeg expects for the right
# playback direction.
pmm = PlateModelManager()
model = pmm.get_model("Zahirovic2022", data_dir="./gplately_data")
recon = gplately.PlateReconstruction(
    rotation_model=model.get_rotation_model(),
    topology_features=model.get_topologies(),
    static_polygons=model.get_static_polygons(),
)
engine = gplately.PygmtPlotEngine()


def render_pmm_frame(t):
    """Render one plate-motion frame at reconstruction time `t` (Ma)."""
    gplot = gplately.PlotTopologies(
        plate_reconstruction=recon,
        coastlines=model.get_coastlines(),
        continents=model.get_continental_polygons(),
        COBs=model.get_COBs(), time=t)
    fig = pygmt.Figure()
    fig.basemap(region="d", projection="W0/20c",
                frame=["af", f'+t{t} Ma'])
    engine.plot_geo_data_frame(fig, gplot.get_continents(),
                               fill="gray95", pen="0.2p,gray30")
    engine.plot_geo_data_frame(fig, gplot.get_coastlines(),
                               pen="0.3p,gray20")
    engine.plot_geo_data_frame(fig, gplot.get_all_topological_sections(),
                               pen="0.5p,gray50")
    engine.plot_geo_data_frame(fig, gplot.get_ridges(), pen="0.5p,red")
    (_tl, _tr) = gplot.get_subduction_direction()
    engine.plot_subduction_zones(fig, _tl, _tr, color="blue")
    return fig


# Preview: one frame at 120 Ma (mid-Cretaceous, classic Pangaea-breakup state)
preview_pmm = render_pmm_frame(120)
preview_pmm.show(width=700)
display(HTML('<div style="height:1cm"></div>'))

In [ ]:
# Cell 2b — render all 51 plate-motion frames + encode MP4
START, END, STEP = 250, 0, 5    # Ma
for i, t in enumerate(range(START, END - 1, -STEP)):
    fig = render_pmm_frame(t)
    fig.savefig(FRAMES_PMM / f"frame_{i:04d}.png", dpi=120)
    print(f"  frame {i:>3}/{(START-END)//STEP}  ({t} Ma)")
print("Plate-motion frames done.")

## 3. Quick-look GIF with `imageio`

Drop-in for a README or a Slack thread. Not compressed efficiently (each frame is a full image) so the file is large — use MP4 for anything bigger than ≈50 frames.

In [ ]:
# Cell 3 — assemble GIF
import imageio.v3 as iio
from PIL import Image

GIF_OUT = VIDEOS / "plate_motion.gif"
frames = sorted(FRAMES_PMM.glob("frame_*.png"))   # 250 Ma → 0 Ma order

# pyGMT's tight crop produces frames with slightly different sizes from
# one t to the next (the visible boundary network changes shape).
# imageio.imwrite ends up calling np.stack which needs identical shapes,
# so pad every frame to a common (max_w, max_h) with a white background
# before stitching.
opened = [Image.open(p).convert("RGB") for p in frames]
max_w = max(im.size[0] for im in opened)
max_h = max(im.size[1] for im in opened)

padded = []
for im in opened:
    canvas = Image.new("RGB", (max_w, max_h), color="white")
    canvas.paste(im, ((max_w - im.size[0]) // 2, (max_h - im.size[1]) // 2))
    padded.append(canvas)

iio.imwrite(GIF_OUT,
            [np.array(im) for im in padded],
            duration=120, loop=0)
print(f"Wrote {GIF_OUT}  ({GIF_OUT.stat().st_size / 1e6:.1f} MB, "
      f"padded to {max_w}×{max_h})")

## 4. Cross-platform MP4 with `ffmpeg`

Key encoder flags (the same trio is used for all three MP4s in this notebook):

- `-c:v libx264` — H.264 video codec, available on every platform.
- `-pix_fmt yuv420p` — the pixel format every consumer player supports (QuickTime, Windows Media, browsers). Without this flag ffmpeg often defaults to `yuv444p`, which plays in some browsers but not in QuickTime.
- `-movflags +faststart` — moves the metadata to the head of the file so the MP4 can stream over the web.

In [ ]:
# Cell 4 — plate motion: encode as cross-platform MP4
def encode_mp4(frames_dir: Path, out: Path, framerate: int = 10) -> None:
    '''Encode every frame_*.png in `frames_dir` as a cross-platform MP4.

    The `-vf pad=ceil(iw/2)*2:ceil(ih/2)*2` filter rounds odd-numbered
    frame dimensions up to the next even pixel — libx264 + yuv420p
    requires even width AND height, and pyGMT's tight bounding-box
    crop sometimes produces odd dims (e.g. 619×713) that would
    otherwise make ffmpeg bail with returncode 1.
    '''
    cmd = [
        "ffmpeg", "-y",
        "-framerate", str(framerate),
        "-pattern_type", "glob",
        "-i", str(frames_dir / "frame_*.png"),
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2:color=white",
        "-movflags", "+faststart",
        "-crf", "20",
        str(out),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("ffmpeg failed — last 2 KB of stderr:\n")
        print(result.stderr[-2000:])
        raise subprocess.CalledProcessError(
            result.returncode, cmd, result.stdout, result.stderr)
    print(f"Wrote {out}  ({out.stat().st_size / 1e6:.1f} MB)")

encode_mp4(FRAMES_PMM, VIDEOS / "plate_motion.mp4", framerate=10)

## 5. Lithospheric thickness — rotating-globe animation

Afonso et al. ([*GJI* 2019](https://academic.oup.com/gji/article/217/3/1602/5370085)) inverted multiple geophysical datasets to produce a global reference model of the lithosphere and upper mantle. We use their lithospheric thickness grid (`data/lithospheric_thickness_Afonso2019.nc`) and render it as a rotating orthographic globe: viewing longitude sweeps **0° → 360°** in 10° steps over 36 frames.

Recipe per frame:

1. `pygmt.makecpt(cmap="dem4", series=[0, 250, 5], background="o")` — the GMT `dem4` topographic palette runs from cool greens (thinnest lithosphere, ≈0 km) through tans to bright tops (thickest cratonic roots). `background="o"` clips any value > 200 km to the same colour as 200, so a few outliers don't wash out the entire scale.
2. `fig.grdimage(grid_km, projection=f"G{lon}/0/12c", region="g", cmap=True)`
3. `fig.coast(shorelines="0.3p,white")` — overlay thin white coast lines.
4. `fig.colorbar(position="JBC+w10c/0.35c+h+o0/1c", frame='af+l"Lithospheric thickness (km)"')` — horizontal colourbar below the globe.
5. `fig.savefig(frames_dir / f"frame_{i:04d}.png", dpi=130)`.

pyGMT auto-crops each saved PNG tightly to the figure's bounding box, so the resulting MP4 has no wasted margins.

In [ ]:
# Cell 5 — litho-thickness rotating-globe setup + preview frame
LITHO_GRID_PATH = DATA_ROOT / "lithospheric_thickness_Afonso2019.nc"
if not LITHO_GRID_PATH.exists():
    raise FileNotFoundError(f"missing: {LITHO_GRID_PATH}")

# Load and convert metres → km (the file is in metres; km is friendlier)
litho_km = xr.open_dataset(LITHO_GRID_PATH)["z"] / 1000.0
print(f"Litho thickness: {float(litho_km.min()):.1f} … "
      f"{float(litho_km.max()):.1f} km")

NFRAMES_LITHO = 72   # 5° viewing-longitude step (smooth, no jump)


def render_litho_frame(lon):
    """Render one litho-thickness rotating-globe frame at viewing
    longitude `lon` (degrees). Returns a pygmt.Figure."""
    fig = pygmt.Figure()
    pygmt.makecpt(cmap="dem4", series=[0, 250, 5], background="o")
    fig.grdimage(litho_km,
                 projection=f"G{lon:.2f}/0/12c", region="g",
                 cmap=True, shading="+a315+nt1.5")
    fig.coast(shorelines="0.3p,white")
    fig.colorbar(position="JBC+w10c/0.35c+h+o0/1c",
                 frame='a50f25+lLithospheric thickness (km)')
    return fig


# Preview: looking at Greenwich (lon = 0°) — shows Africa, Europe, the
# East-European craton, and the spine of the mid-Atlantic ridge
preview_litho = render_litho_frame(0)
preview_litho.show(width=600)
display(HTML('<div style="height:1cm"></div>'))

In [ ]:
# Cell 5b — render all 36 rotating-globe frames + encode MP4
for i in range(NFRAMES_LITHO):
    lon = i * 360.0 / NFRAMES_LITHO
    fig = render_litho_frame(lon)
    fig.savefig(FRAMES_LITHO / f"frame_{i:04d}.png", dpi=130)
    print(f"  frame {i:>2}/{NFRAMES_LITHO}  (lon = {lon:6.2f}°)")
encode_mp4(FRAMES_LITHO, VIDEOS / "litho_thickness.mp4", framerate=6)

## 6. Reconstructed lithospheric thickness — 0 → 60 Ma

The rotating globe above is a pure pyGMT exercise — it doesn't actually use GPlately. To bring the plate-tectonic engine back into the picture, we now **reconstruct** the present-day Afonso 2019 lithospheric-thickness grid back through the last 60 Myr.

GPlately's `Raster` class makes the recipe a one-liner per frame:

1. We build a `gplately.Raster` from the present-day litho-thickness array, telling it which `PlateReconstruction` to use (the Zahirovic 2022 model from earlier cells).
2. At each reconstruction time `t`, we call `raster.reconstruct(time=t)`. Under the hood that uses the model's **static polygons** to segment the present-day grid into plate-mounted pieces, then applies each plate's finite rotation to rotate that piece back to its paleo-position. The result is a global grid at time `t` showing where the present-day lithospheric thickness *would have been* under the Z22 reconstruction.
3. We render the reconstructed grid in pyGMT (Robinson global view, cubehelix CPT, time label) with the reconstructed coastlines from `gplot` at that age overlaid in white.

**Caveats.** The reconstruction assumes lithospheric thickness is *plate-mounted* — i.e. each plate carries its mantle root with it as it moves. That's a defensible first-order assumption for old, cold cratonic mantle (which deforms only on much longer timescales than the 60-Myr window), but it ignores active lithospheric thinning above mantle plumes, lithospheric thickening at collisions, and steady-state cooling of newly created oceanic lithosphere. The video is therefore a *kinematic-only* visualisation of where the present-day lithosphere came from, not a forward model of how it formed.

In [ ]:
# Cell 7 — reconstructed litho-thickness setup + preview frame
#
# Build the Raster once. `plate_reconstruction=recon` was created
# above (in the plate-motion section); reuse it so we share the
# same Zahirovic2022 rotation model and static polygons.
#
# gplately.Raster reads the lat / lon axes straight off the xarray
# DataArray — no need for .values or numpy intermediates anywhere.
# The same pattern works if you pass a netcdf file path instead.
litho_raster = gplately.Raster(
    data=litho_km,
    plate_reconstruction=recon,
    time=0.0,
)

# Present-day seafloor age grid from Zahirovic 2022. The age tells us
# WHEN each oceanic grid cell was created at a mid-ocean ridge — i.e.
# the deepest time we can validly reconstruct it back to. At each
# render time t, any cell with age < t is oceanic crust that did not
# yet exist and should be masked. Static polygons alone cannot do
# this masking because they only know the present-day plate, not the
# crustal age. (Following the pyGPlates / EarthByte recipe for
# combining static polygons + age grid for global grid reconstruction.)
age_file_0 = model.get_raster("AgeGrids", time=0)
present_age = (xr.open_dataarray(str(age_file_0))
               if str(age_file_0).endswith(".nc")
               else xr.open_dataset(str(age_file_0))["z"])
# Re-grid to the same lat/lon as the litho-thickness raster so the
# mask aligns pixel-for-pixel.
present_age = present_age.interp(lat=litho_km.lat, lon=litho_km.lon)
print(f"Present-day age grid: {float(present_age.min()):.1f} … "
      f"{float(present_age.max()):.1f} Ma, "
      f"NaN {float(np.isnan(present_age).mean())*100:.1f}% (continents)")

# Per-frame `gplot` to pull reconstructed coastlines for each time.
gplot_rc = gplately.PlotTopologies(
    plate_reconstruction=recon,
    coastlines=model.get_coastlines(),
    continents=model.get_continental_polygons(),
    COBs=model.get_COBs(),
    time=0,
)
engine_rc = gplately.PygmtPlotEngine()

TIMES_RC = list(range(0, 61, 2))    # 0, 2, 4, …, 60 Ma → 31 frames

# Diagnostic: reconstruct + save_to_netcdf4 sanity check at t=0 and t=30.
import numpy as _np
print("=== Diagnostic ===")
for _t in (0, 30):
    _r = litho_raster if _t == 0 else litho_raster.reconstruct(time=float(_t))
    _data = _r.data
    _nan_pct = _np.isnan(_data).mean() * 100
    if _nan_pct < 100:
        _vmin = _np.nanmin(_data); _vmax = _np.nanmax(_data)
        _range = f"{_vmin:.1f}–{_vmax:.1f} km"
    else:
        _range = "all NaN"
    _tmp = Path(f"./_diag_t{_t:03d}.nc")
    _r.save_to_netcdf4(str(_tmp))
    _saved = _tmp.exists() and _tmp.stat().st_size > 0
    print(f"  t = {_t:>2} Ma  → shape {_data.shape}  "
          f"NaN {_nan_pct:5.1f}%  {_range}  "
          f"netcdf saved={_saved}, {_tmp.stat().st_size/1024 if _saved else 0:.0f} kB")
    if _saved:
        _tmp.unlink()
print("=== end diagnostic ===")


def render_recon_frame(t):
    """Render one reconstructed-litho-thickness frame at time `t` (Ma).

    Recipe (per the pyGPlates static-polygons + age-grid pattern):
      1. Reconstruct the present-day litho-thickness grid back to t
         using `gplately.Raster.reconstruct` (static polygons partition
         each cell by plate ID, then rotate by the plate finite
         rotation 0 → t).
      2. Mask any cell where the present-day seafloor age is younger
"
         than t — that's oceanic crust that did not yet exist at
         time t. Continental cells (age = NaN) are kept.
      3. Save to a small temporary netcdf file and hand to pyGMT.
    """
    if t == 0:
        recon_obj = litho_raster
    else:
        recon_obj = litho_raster.reconstruct(time=float(t))

    # Build an xarray DataArray of the reconstructed thickness so we
    # can apply the age mask cleanly.
    recon_da = xr.DataArray(
        recon_obj.data, dims=("lat", "lon"),
        coords={"lat": litho_km.lat, "lon": litho_km.lon},
        name="litho_thickness_km",
    )
    if t > 0:
        # Mask oceanic crust younger than t — it didn't exist yet.
        # NaN in the age grid means continental and is kept.
        young_ocean = (present_age < t).fillna(False)
        recon_da = recon_da.where(~young_ocean)

    TEMP_GRID = Path(f"./_recon_t{int(t):03d}.nc")
    recon_da.to_netcdf(str(TEMP_GRID))

    fig = pygmt.Figure()
    pygmt.makecpt(cmap="dem4", series=[0, 250, 5], background="o")
    # NOTE: dropped `shading="+a315+nt1.5"` — the on-the-fly gradient
    # computation appears to silently fail on grids with large NaN
    # regions (which is what reconstructed-litho looks like, since
    # static polygons only cover continents). Plain grdimage works.
    fig.grdimage(str(TEMP_GRID), projection="W0/18c", region="d",
                 cmap=True, nan_transparent=True)
    gplot_rc.time = t
    engine_rc.plot_geo_data_frame(fig, gplot_rc.get_coastlines(),
                                  pen="0.3p,white")
    fig.colorbar(position="JBC+w12c/0.35c+h+o0/1c",
                 frame='a50f25+lLithospheric thickness (km)')
    fig.text(x=-178, y=80, text=f"{t} Ma",
             justify="TL", font="22p,Helvetica-Bold,white",
             fill="black@40", offset="0.2c/-0.2c")
    TEMP_GRID.unlink()  # tidy up the temp grid
    return fig


# Preview: 30 Ma — India well underway on its dash north, Atlantic
# noticeably narrower than today, Drake Passage opening.
preview_recon = render_recon_frame(30)
preview_recon.show(width=700)
display(HTML('<div style="height:1cm"></div>'))

In [ ]:
# Cell 7b — render all 31 reconstructed-litho frames + encode MP4
for i, t in enumerate(TIMES_RC):
    fig = render_recon_frame(t)
    fig.savefig(FRAMES_LITHO_RC / f"frame_{i:04d}.png", dpi=130)
    print(f"  frame {i:>2}/{len(TIMES_RC)}  (t = {t:>2} Ma)")
encode_mp4(FRAMES_LITHO_RC,
           VIDEOS / "litho_thickness_reconstructed.mp4",
           framerate=6)

## What the three animations tell us

**Plate motion** — speed contrast. Some plate pairs (India–Asia, Africa–Eurasia) are spectacularly fast through the Cenozoic; others (Antarctica, the West-African craton) barely move. This visual contrast is the easiest way to build intuition for which plates are kinematically active and which are passively along for the ride.

**Lithospheric thickness** — the longest-lived signal in the lithosphere. Old cratons (West Africa, Siberia, the Canadian Shield, the Australian and Antarctic cores) glow dark (>200 km thick), while young ocean basins and active margins (East African Rift, mid-ocean ridges, the western US) read lighter (<80 km). Watch the rotation cycle past each continent to see how the thickest cores anchor every craton.

**Reconstructed lithospheric thickness (0–60 Ma)** — the cratonic roots in motion. Watch the slow march of the West-African and Australian-Antarctic cratons (dark) and the dramatic flight of India (also dark — its cratonic root is one of the thickest on Earth) as the Tethys closes. The reconstructed grid stays bound to each plate so the cratonic geometry comes along for the ride; this is the kinematic part of the lithospheric story, isolated from any thermal evolution.

## Extend this

- **Higher resolution.** Bump `dpi=130` in cells 5 / 6 to 200 and rerun. Output MP4s will be larger but visibly sharper at 1080p.
- **Combine the videos side-by-side** via `ffmpeg`'s `hstack` filter for a small demo reel.
- **Drape an age grid behind the plate-motion frames** for an "ocean-floor age" version.
- **Use `gmt movie` for the polished version** with built-in progress bar and time labels — pass the same per-frame Python logic from a main_script.sh that `subprocess`-invokes a render-one-frame Python script. See [the GMT movie docs](https://docs.generic-mapping-tools.org/latest/movie.html).

## References

- Mather, B.R., Müller, R.D., Zahirovic, S., Cannon, J., Chin, M., Ilano, L., Wright, N.M., Alfonso, C., Williams, S., Tetley, M. & Merdith, A. (2024). Deep time spatio-temporal data analysis using GPlately. *Geosci. Data J.* 11, 3-10. https://doi.org/10.1002/gdj3.185
- Tian, D., Uieda, L., Leong, W.J., Fröhlich, Y., Schlitzer, W., Grund, M., Jones, M., Toney, L., Yao, J., Magen, Y., Materna, K., Belem, A., Newton, T., Anant, A., Ziebarth, M., Quinn, J., Wessel, P. (2024). PyGMT: A Python interface for the Generic Mapping Tools. *Zenodo*. https://doi.org/10.5281/zenodo.13679085
- Wessel, P., Luis, J.F., Uieda, L., Scharroo, R., Wobbe, F., Smith, W.H.F. & Tian, D. (2019). The Generic Mapping Tools version 6. *Geochem. Geophys. Geosys.* 20, 5556-5564. https://doi.org/10.1029/2019GC008515
- Chin, M., Mather, B.R. & Müller, R.D. (2024). Plate Model Manager: A Python package for downloading and managing published plate-reconstruction models. *Zenodo*. https://github.com/michaelchin/plate-model-manager
- Zahirovic, S., Eleish, A., Doss, S., Pall, J., Cannon, J., Pistone, M., Tetley, M.G., Young, A. & Müller, R.D. (2022). Subduction and carbonate platform interactions. *Geoscience Data Journal* 9, 371-383. https://doi.org/10.1002/gdj3.146
- Afonso, J.C., Salajegheh, F., Szwillus, W., Ebbing, J. & Gaina, C. (2019). A global reference model of the lithosphere and upper mantle from joint inversion of surface waves and gravity. *Geophys. J. Int.* 217, 1602-1628. https://doi.org/10.1093/gji/ggz094
- FFmpeg Developers (2024). FFmpeg multimedia framework. https://ffmpeg.org